# Breast Cancer Wisconsin Diagnostic: Model Training and Testing

**Goal:** classify diagnosis as Benign or Malignant using 30 numerical measurements.

This notebook:
1. Loads `data.csv`
2. Cleans identifier/empty columns
3. Splits data into train/test sets
4. Trains a scaled RBF SVC
5. Reports accuracy, ROC-AUC, confusion matrix and cross-validation
6. Saves `model.joblib` and `metrics.json`

> Educational use only. This is not a medical diagnostic tool.

In [ ]:
import json
import joblib
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

DATA_PATH = Path("data.csv")
df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
display(df.head())

In [ ]:
df = df.drop(columns=["id", "Unnamed: 32"], errors="ignore").dropna(subset=["diagnosis"])
y = df.pop("diagnosis").map({"B": 0, "M": 1})
X = df.astype(float)
print("Features:", X.shape[1])
print("Target distribution:")
print(y.map({0: "Benign", 1: "Malignant"}).value_counts())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", SVC(C=1.0, kernel="rbf", probability=True, random_state=42))
])
model.fit(X_train, y_train)
pred = model.predict(X_test)
proba = model.predict_proba(X_test)[:, 1]
print(f"Test accuracy: {accuracy_score(y_test, pred):.2%}")
print(f"Test ROC-AUC: {roc_auc_score(y_test, proba):.4f}")
print("Confusion matrix (Benign, Malignant):")
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, target_names=["Benign", "Malignant"]))

In [ ]:
cv_scores = cross_val_score(
    model, X, y,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring="accuracy"
)
print(f"5-fold CV accuracy: {cv_scores.mean():.2%} +/- {cv_scores.std():.2%}")

In [ ]:
metrics = {
    "model": "StandardScaler + RBF SVC",
    "test_accuracy": float(accuracy_score(y_test, pred)),
    "test_roc_auc": float(roc_auc_score(y_test, proba)),
    "five_fold_cv_accuracy_mean": float(cv_scores.mean()),
    "five_fold_cv_accuracy_std": float(cv_scores.std()),
}
bundle = {
    "model": model,
    "feature_names": list(X.columns),
    "label_map": {0: "Benign", 1: "Malignant"},
    "metrics": metrics,
}
joblib.dump(bundle, "model.joblib")
Path("metrics.json").write_text(json.dumps(metrics, indent=2))
print("Saved model.joblib and metrics.json")
print(metrics)